<div style="font-size: 0.85em;">
  <h3>Mini-Project: Structured RAG Output with Citations</h3>
  <p>In this mini-project, we will simulate a RAG pipeline that:</p>
  <ul>
    <li>Takes a user question.</li>
    <li>Retrieves relevant documents from a small in-memory list (no vector database yet).</li>
    <li>Uses the retrieved text as context.</li>
    <li>Asks the LLM to generate an answer.</li>
    <li>Uses <code>PydanticOutputParser</code> to return a structured object with:
      <ul>
        <li><code>question</code> – the original question.</li>
        <li><code>answer</code> – the generated answer.</li>
        <li><code>source</code> – the document used to answer.</li>
        <li><code>confidence</code> – a placeholder score.</li>
      </ul>
    </li>
  </ul>
  <p>We’ll also add <code>OutputFixingParser</code> to handle any malformed model output.</p>
</div>

Imports and Define Response Model

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain.output_parsers import OutputFixingParser
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [2]:
# Load environment variables
load_dotenv()

# Define the structured output model for the RAG response
class RAGResponse(BaseModel):
    question: str = Field(description='The original user question')
    answer: str = Field(description='The generated answer')
    source: str = Field(description='The source document used')
    confidence: float = Field(description='Confidence score between 0 and 1')
    
print('Imports and model defined.')

Imports and model defined.


Simulated Document Store and Retriever


In [4]:
# Define a small in-memory document store (simulated knowledge base)
# Each document has text content and a source name.
documents = [
    {
        'text': 'RAG is a technique that combines retrieval of relevant documents with language generation.',
        'source': 'doc1_rag_intro.txt'
    },
    {
        'text':  'Vector databases store embeddings and perform fast similarity search.',
        'source': 'doc2_vector_db.txt' 
    },
    {
        'text': 'LangChain provides tools for building RAG pipelines easily.',
        'source': 'doc3_langchain.txt'
    }   
]

# Simulate a simple retriever: return the first document 
def retrieve(query: str) -> dict:
    return documents[0]

# Test the retriever
sample = retrieve('What is RAG?')
print(f"Retrieved sources: {sample['source']}")
print(f"Retrieved text: {sample['text']}")

Retrieved sources: doc1_rag_intro.txt
Retrieved text: RAG is a technique that combines retrieval of relevant documents with language generation.


Create Parsers and Prompt

In [7]:
# Create the Pydantic parser for RAGResponse
parser = PydanticOutputParser(pydantic_object=RAGResponse)

# Create a fixing parser to auto-correct invalid output
fixing_parser = OutputFixingParser.from_llm(
    parser=parser,
    llm=ChatOpenAI(model='gpt-4o-mini', temperature=0)
)

# Create chat model
llm=ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Build the prompt template.
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Answer the question using only the provided context.'),
    ('human',
     'Context: {context}\n'
     'Source: {source}\n\n'
     'Question: {question}\n\n'
     '{format_instructions}'
     )
])

# Pre-fill the format instructions to only pass context, source, and question later
prompt = prompt.partial(format_instructions=parser.get_format_instructions())
print('Parsers and prompt created.')

Parsers and prompt created.


Build and Run the RAG Chain

In [10]:
# Build the chain:
# prompt -> model -> fixing_parser
chain = prompt | llm | fixing_parser

# Simulate a user question
question = 'What do you understand by RAG?'

# Retrieve a document
doc = retrieve(question)

# Invoke the chain with the retrieved context, source, and question
result = chain.invoke({
    'context': doc['text'],
    'source': doc['source'],
    'question': question
})

# Print the structured result
print('Result:')
print(result)
print('Type:', type(result).__name__)

Result:
question='What do you understand by RAG?' answer='RAG is a technique that combines retrieval of relevant documents with language generation.' source='doc1_rag_intro.txt' confidence=0.95
Type: RAGResponse
